# The kernel trick

_Machine Learning_

**Draw curved boundaries without ever building the curved features.**

A straight line cannot separate every dataset. Some need curves.

     One fix: map the data into a higher dimension where a straight line works again.

---

This notebook builds the kernel-trick demo up one step at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

We'll build a dataset that *no* straight line can separate, then compare a linear SVM against an RBF-kernel SVM on it. We build it in three steps: (1) make the rings and split them, (2) fit both models, (3) compare their accuracy.

### Step 1 — Make two concentric rings

`make_circles` draws an inner ring of one class surrounded by an outer ring of the other. No straight line can separate a circle from the ring around it, which is exactly the situation the kernel trick is built for. We hold out 25% of the points as a test set.

In [ ]:
import numpy as np
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

# Two concentric rings: not linearly separable.
X, y = make_circles(n_samples=400, noise=0.08, factor=0.4, random_state=0)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)
print("train:", Xtr.shape[0], " test:", Xte.shape[0])

### Step 2 — Fit a linear SVM and an RBF SVM

The **linear** SVM can only draw a straight boundary in the original 2-D space. The **RBF** (radial basis function) kernel implicitly maps the points into a much higher-dimensional space where a straight boundary *can* separate the rings — without ever computing those coordinates explicitly. That implicit mapping is the kernel trick.

In [ ]:
from sklearn.svm import SVC

linear = SVC(kernel="linear").fit(Xtr, ytr)
rbf = SVC(kernel="rbf", gamma=1.0).fit(Xtr, ytr)

### Step 3 — Compare test accuracy

We score each fitted model on the held-out test set. The linear kernel should struggle on the rings while the RBF kernel separates them cleanly — a direct demonstration of what the kernel trick buys you.

In [ ]:
print("linear kernel test accuracy:", round(linear.score(Xte, yte), 3))
print("RBF    kernel test accuracy:", round(rbf.score(Xte, yte), 3))

## Visualize the data & results

_On the full 30-feature breast-cancer data, does an RBF kernel beat a plain linear SVM?_

Now we move to a real, high-dimensional dataset and visualise both the data and the accuracy comparison. We build it in three steps.

### Step 1 — Standardise the data and split it

The 30 breast-cancer features live on very different scales, so we **standardise** them before training — SVMs with an RBF kernel are sensitive to feature scale. We then split off a 25% test set to score on later.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

bc = load_breast_cancer()

# Put every feature on the same scale.
X = StandardScaler().fit_transform(bc.data)
y = bc.target

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)
print("train:", Xtr.shape[0], " test:", Xte.shape[0])

### Step 2 — Project to 2-D and plot the two tumour classes

We use **PCA** to compress the 30 features down to 2 so we can draw the points, then colour them by class. This is the left-hand panel — note how much the malignant and benign clouds overlap, which is why this is a hard problem.

In [ ]:
from sklearn.decomposition import PCA

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Compress 30 features down to 2 for plotting.
P = PCA(n_components=2, random_state=0).fit_transform(X)

for label, color, name in [(0, "#ff7b72", "malignant"), (1, "#7ee787", "benign")]:
    pts = P[y == label]
    ax1.scatter(pts[:, 0], pts[:, 1], c=color, label=name, edgecolor="k")
ax1.set_xlabel("PCA component 1")
ax1.set_ylabel("PCA component 2")
ax1.set_title("Breast-cancer tumors in 2-D")
ax1.legend()

### Step 3 — Compare linear vs RBF on the full feature set

We train both kernels on all 30 standardised features (not the 2-D projection) and score them on the test set. The right-hand bar chart shows whether the RBF kernel's curved boundaries actually help on this real dataset, then we draw both panels.

In [ ]:
from sklearn.svm import SVC

lin = SVC(kernel="linear").fit(Xtr, ytr).score(Xte, yte)
rbf = SVC(kernel="rbf", gamma="scale").fit(Xtr, ytr).score(Xte, yte)

ax2.bar(["linear kernel", "RBF kernel"], [lin, rbf], color=["#4ea1ff", "#7ee787"])
ax2.set_title("Test accuracy: linear vs RBF")
plt.show()